In [ ]:
!pip install xgboost scikit-learn pandas numpy

: 

In [ ]:
# Optional: Ensure xgboost and dependencies exist (helpful in Colab). If pip install is disallowed in this environment you'll see a benign warning.
import sys, os
print('Python', sys.version.split()[0])
try:
    import xgboost as xgb
    print('xgboost', xgb.__version__)
except Exception as e:
    print('xgboost import failed:', e)
    print('Attempting to (re)install xgboost...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost'])
    import importlib
    importlib.invalidate_caches()
    try:
        import xgboost as xgb
        print('xgboost', xgb.__version__)
    except Exception as e2:
        print('Still failed to import xgboost:', e2)
# Use Agg backend to ensure plots save in headless environments
import matplotlib
matplotlib.use('Agg')
# Core ML/data libs (kept here to consolidate imports)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [ ]:
# Load Kaggle Titanic data (Colab-friendly upload fallback)
import os
if not os.path.exists('train.csv') or not os.path.exists('test.csv'):
    try:
        from google.colab import files
        print('train.csv or test.csv not found. Use the file upload dialog to upload them now.')
        uploaded = files.upload()  # interactive upload in Colab
        print('Uploaded files:', list(uploaded.keys()))
    except Exception:
        print('train.csv/test.csv not found in working dir. Please upload them to the notebook or mount Drive.')
if not os.path.exists('train.csv') or not os.path.exists('test.csv'):
    raise FileNotFoundError('train.csv and/or test.csv not found in the working directory. Upload them in Colab or place in the working dir.')
train_df = pd.read_csv('train.csv')  # has Survived column
test_df  = pd.read_csv('test.csv')   # no Survived column
print(train_df.head())
print(train_df.info())

In [ ]:
# Helper: save notebook and `outputs/` to Drive if running in Colab and Drive is mounted
import os
import shutil
def save_to_drive(drive_dir='/content/drive/MyDrive/XGBoostLearning_outputs'):
    try:
        if not os.path.exists('/content/drive'):
            raise RuntimeError('Drive not mounted')
        os.makedirs(drive_dir, exist_ok=True)
        nb_src = 'XGBoostLearning.ipynb'
        if os.path.exists(nb_src):
            shutil.copy(nb_src, os.path.join(drive_dir, os.path.basename(nb_src)))
        if os.path.exists('outputs'):
            dst = os.path.join(drive_dir, 'outputs')
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree('outputs', dst)
        print('Copied notebook and outputs to', drive_dir)
    except Exception as e:
        print('Could not save to Drive (not in Colab or Drive not mounted).', e)

**Colab Guidance**  
- If you are running on Colab: upload `train.csv`/`test.csv` using the file upload dialog or mount Google Drive using the snippet below.  
- To mount drive (one-time):  
```python
from google.colab import drive
drive.mount('/content/drive')
# then read files from /content/drive/MyDrive/path/to/train.csv
```
- I also provide a helper cell below to copy the notebook and `outputs/` to your Drive if mounted.

In [ ]:
# Preprocessing helper defined near the top so it's available to later cells
def preprocess(df, is_train=True):
    df = df.copy()
    # Defensive handling for missing columns
    if 'Name' not in df.columns:
        df['Name'] = ''
    df['Title'] = df['Name'].str.extract(',\s*([^\.]+)\.', expand=False).str.strip().fillna('Other')
    common_titles = ['Mr', 'Mrs', 'Miss', 'Master']
    df['Title'] = df['Title'].where(df['Title'].isin(common_titles), 'Other')
    df['FamilySize'] = df.get('SibSp', 0) + df.get('Parch', 0) + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    # Fill numeric columns if missing
    if 'Age' in df.columns:
        df['Age'] = df['Age'].fillna(df['Age'].median())
    else:
        df['Age'] = 0
    if 'Fare' in df.columns:
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    else:
        df['Fare'] = 0.0
    if 'Embarked' in df.columns:
        df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode().iloc[0])
    if 'Sex' in df.columns:
        df['Sex'] = df['Sex'].map({'male': 1, 'female': 0}).fillna(0).astype(int)
    else:
        df['Sex'] = 0
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'Title']
    for c in features:
        if c not in df.columns and c not in ['Title', 'FamilySize', 'IsAlone']:
            raise KeyError(f'Missing expected column in input DataFrame: {c}')
    X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'Title']].copy()
    X = pd.get_dummies(X, columns=['Title'], drop_first=True)
    if 'Embarked' in df.columns:
        emb = pd.get_dummies(df['Embarked'], prefix='Emb', drop_first=True)
        X = pd.concat([X, emb], axis=1)
    return X

In [ ]:
# Run preprocessing on train/test and inspect shapes
X_full = preprocess(train_df)
y_full = train_df['Survived']
X_test_final = preprocess(test_df, is_train=False)
print('Train features shape:', X_full.shape)
print('Test features shape:', X_test_final.shape)
print('Feature columns:', X_full.columns.tolist())

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

print("Train:", X_train.shape, "Val:", X_val.shape)


In [ ]:
model = XGBClassifier(
    n_estimators=300,          # number of trees
    learning_rate=0.05,       # smaller step size
    max_depth=4,              # tree depth
    subsample=0.8,            # row sampling
    colsample_bytree=0.8,     # feature sampling
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)


In [ ]:
y_val_pred = model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))


In [ ]:
importances = model.feature_importances_
feature_names = np.array(X_train.columns)

sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 6))
plt.bar(range(len(importances)), importances[sorted_idx])
plt.xticks(range(len(importances)), feature_names[sorted_idx], rotation=90)
plt.title("XGBoost Feature Importance (Titanic)")
plt.tight_layout()
import os
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/feature_importance_train.png', bbox_inches='tight')
plt.show()

In [ ]:
final_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

final_model.fit(X_full, y_full)

test_pred = final_model.predict(X_test_final)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300, 400],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric="logloss"),
    param_distributions=params,
    n_iter=10,
    scoring="accuracy",
    cv=5,
    verbose=1,
    n_jobs=-1
)

search.fit(X_full, y_full)
print(search.best_params_)


In [ ]:
best_params = {
    'subsample': 0.8,
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.01,
    'colsample_bytree': 1.0,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_jobs': -1,
    'random_state': 42
}

best_model = XGBClassifier(**best_params)

best_model.fit(X_full, y_full)


In [ ]:
test_pred = best_model.predict(X_test_final)


In [ ]:
from sklearn.metrics import confusion_matrix
# matplotlib and numpy already imported at top-level
# Predictions on validation set
y_val_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val, y_val_pred)
classes = ["Not Survived (0)", "Survived (1)"]

fig, ax = plt.subplots()
im = ax.imshow(cm, interpolation='nearest')
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=classes,
    yticklabels=classes,
    ylabel='True label',
    xlabel='Predicted label',
    title='Confusion Matrix (Validation)',
 )

# Write numbers inside the squares
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black"
        )

plt.tight_layout()
import os
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=pd.Index(classes, name="True"),
    columns=pd.Index(classes, name="Predicted")
)
cm_df


In [ ]:
from sklearn.metrics import classification_report

report_dict = classification_report(y_val, y_val_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).T
report_df


In [ ]:
from xgboost import plot_importance
import os
plt.figure()
plot_importance(best_model, max_num_features=15)
plt.title("Top 15 Feature Importances (XGBoost)")
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
importances = best_model.feature_importances_
feature_names = X_train.columns

fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fi_df.head(20)   # top 20 features

In [ ]:
from sklearn.metrics import roc_curve, auc
y_val_proba = best_model.predict_proba(X_val)[:, 1]
fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")  # random baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost (Validation)")
plt.legend(loc="lower right")
plt.grid(True)
import os
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/roc_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# Rebuild a validation DataFrame (if you still have original train_df)
val_df = train_df.loc[X_val.index].copy()

val_df["y_true"] = y_val.values
val_df["y_pred"] = y_val_pred
val_df["y_proba"] = y_val_proba

# Show a few rows where model was wrong
val_df[val_df["y_true"] != val_df["y_pred"]].head(10)[
    ["PassengerId", "Sex", "Pclass", "Age", "Fare", "y_true", "y_pred", "y_proba"]
]


In [ ]:
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ["0–0.2", "0.2–0.4", "0.4–0.6", "0.6–0.8", "0.8–1.0"]

bucket = pd.cut(y_val_proba, bins=bins, labels=labels, include_lowest=True)

bucket_df = pd.DataFrame({
    "bucket": bucket,
    "y_true": y_val,
    "y_pred": y_val_pred
})

bucket_summary = bucket_df.groupby("bucket").agg(
    count=("y_true", "size"),
    accuracy=("y_true", lambda y: (y.values == bucket_df.loc[y.index, "y_pred"].values).mean())
)

bucket_summary


In [ ]:
# Final: Save outputs and optionally copy to Drive (call explicitly if you want to persist to Drive)
print('Local outputs are in ./outputs (if generated).')
try:
    save_to_drive()
except Exception as e:
    print('Did not save to Drive: mount Drive in Colab and call save_to_drive() if needed.', e)

**Validation Complete ✅**  
- Notebook executed end-to-end locally using the project venv.  
- Key outputs (plots & sample CSVs) were saved to `./outputs/` (see `outputs/feature_importance.png`, `outputs/confusion_matrix.png`, `outputs/roc_curve.png`).  
- If running on Colab and you want these saved to your Drive, mount Drive and run `save_to_drive()` (helper earlier in the notebook).  
